# Phase 6: IMDN XAI final run

This notebook creates the final explanation maps for Phase 6.

The pilot showed that the XAI method works, but its 4x4 heatmaps were coarse. This final run uses fewer selected examples and a finer 8x8 grid, so the explanation maps are clearer for the report.

Important: this notebook does not train a model and does not improve IMDN. It explains how IMDN reacts when parts of the LR input are changed.


## Step 1: Connect Google Drive

The datasets, checkpoints, and output folder are on Drive, so Colab needs Drive access first.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2: Prepare code and packages

This loads the project code and installs the packages needed for the explanation run. `shap` is used for SHAP, and `scikit-learn` is used for the simple LIME-style linear explanation.


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')
if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap>=0.46,<0.50', 'scikit-learn>=1.5,<1.8'], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repository and XAI dependencies ready.')


## Step 3: Settings

A segment is one block of the LR image. The 8x8 grid gives 64 segments, which is clearer than the 16 segments used in the pilot.

The selected cases are not random. They cover smooth regions, fine texture, natural texture, and repeated architectural structure.


In [ ]:
import json
import time
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import numpy as np
import shap
import torch
from PIL import Image, ImageFilter, ImageDraw
from sklearn.linear_model import Ridge
from sklearn.metrics.pairwise import pairwise_distances
from skimage.metrics import peak_signal_noise_ratio

from app.config import dataset_hr_directory, dataset_lr_directory
from app.deep_learning.alignment import align_reconstruction_to_target
from app.deep_learning.checkpoints import download_official_imdn_checkpoint
from app.deep_learning.imdn import imdn_upsample, load_pretrained_imdn
from app.evaluation.experiment import write_results_csv
from app.evaluation.images import load_rgb_image, validate_hr_lr_dimensions
from app.evaluation.metrics import rgb_to_y

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab, select Runtime > Change runtime type > T4 GPU.')

DEVICE = torch.device('cuda')
DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
CHECKPOINT_ROOT = DATA_ROOT / 'checkpoints'
OUTPUT_ROOT = DATA_ROOT / 'results' / 'phase6' / 'imdn_xai_final_v1'
FIGURE_ROOT = OUTPUT_ROOT / 'figures'
METRICS_ROOT = OUTPUT_ROOT / 'metrics'
for directory in (FIGURE_ROOT, METRICS_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

GRID_ROWS = 8
GRID_COLUMNS = 8
LIME_SAMPLES = 96
SHAP_SAMPLES = 96
TARGET_HR_SIZE = 96
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

FINAL_CASES = (
    ('Set5', 'baby.png', 2, 'smooth face and gradual colour regions'),
    ('Set5', 'butterfly.png', 3, 'fine lines and high-frequency texture'),
    ('Set14', 'baboon.png', 4, 'difficult natural texture'),
    ('Urban100', 'img_004.png', 3, 'architectural edges and repetitive structure'),
)

print('GPU:', torch.cuda.get_device_name(DEVICE))
print('Output folder:', OUTPUT_ROOT)
print('Final XAI cases:', len(FINAL_CASES))


## Step 4: Helper functions

These functions are the engine of the explanation. They split the LR image into blocks, hide selected blocks, rerun IMDN, and measure PSNR-Y in the chosen HR region.

In simple terms: hide input block, run IMDN, measure damage. More damage means that block was more important.


In [ ]:
def make_grid_segments(width, height, rows, columns):
    segments = np.zeros((height, width), dtype=np.int32)
    segment_id = 0
    row_edges = np.linspace(0, height, rows + 1, dtype=int)
    column_edges = np.linspace(0, width, columns + 1, dtype=int)
    for row in range(rows):
        for column in range(columns):
            top, bottom = row_edges[row], row_edges[row + 1]
            left, right = column_edges[column], column_edges[column + 1]
            segments[top:bottom, left:right] = segment_id
            segment_id += 1
    return segments


def centre_target_box(image, size):
    width, height = image.size
    box_width = min(size, width)
    box_height = min(size, height)
    left = (width - box_width) // 2
    top = (height - box_height) // 2
    return (left, top, left + box_width, top + box_height)


def draw_target_box(image, box, colour='yellow'):
    copy = image.convert('RGB').copy()
    draw = ImageDraw.Draw(copy)
    for offset in range(3):
        draw.rectangle(
            (box[0] - offset, box[1] - offset, box[2] + offset, box[3] + offset),
            outline=colour,
        )
    return copy


def hr_box_to_lr_box(hr_box, scale):
    return tuple(int(round(value / scale)) for value in hr_box)


def perturb_lr_image(lr_image, segments, keep_mask):
    original = np.asarray(lr_image.convert('RGB'), dtype=np.uint8)
    baseline = np.asarray(lr_image.convert('RGB').filter(ImageFilter.GaussianBlur(radius=3)), dtype=np.uint8)
    hidden = np.isin(segments, np.where(keep_mask == 0)[0])
    output = original.copy()
    output[hidden] = baseline[hidden]
    return Image.fromarray(output, mode='RGB')


def score_target_region(reference_hr, reconstruction, target_box):
    reference_region = np.asarray(reference_hr.crop(target_box).convert('RGB'), dtype=np.float64)
    reconstruction_region = np.asarray(reconstruction.crop(target_box).convert('RGB'), dtype=np.float64)
    reference_y = rgb_to_y(reference_region)
    reconstruction_y = rgb_to_y(reconstruction_region)
    return float(peak_signal_noise_ratio(reference_y, reconstruction_y, data_range=255.0))


def run_imdn_reconstruction(model, lr_image, reference_hr):
    native = imdn_upsample(model, lr_image, DEVICE)
    return align_reconstruction_to_target(native, reference_hr.size).image


def mask_to_score_function(model, reference_hr, lr_image, segments, target_box):
    def score_masks(mask_batch):
        scores = []
        for keep_mask in np.asarray(mask_batch, dtype=int):
            perturbed_lr = perturb_lr_image(lr_image, segments, keep_mask)
            reconstruction = run_imdn_reconstruction(model, perturbed_lr, reference_hr)
            scores.append(score_target_region(reference_hr, reconstruction, target_box))
        return np.asarray(scores, dtype=np.float64)

    return score_masks


## Step 5: Save readable figures

Each final figure shows four things: the LR input, the IMDN output, the selected target region, and the explanation heatmap.

Red blocks mean positive importance: keeping that LR area helped the selected HR region. Blue blocks mean the opposite effect in this local test.


In [ ]:
def importance_to_heatmap(segments, importance):
    heatmap = np.zeros(segments.shape, dtype=np.float64)
    for segment_id, value in enumerate(importance):
        heatmap[segments == segment_id] = value
    max_abs = max(float(np.max(np.abs(heatmap))), 1e-8)
    return heatmap / max_abs


def save_explanation_figure(
    lr_image,
    reconstruction,
    reference_hr,
    segments,
    importance,
    method_name,
    target_box,
    scale,
    output_path,
):
    heatmap_lr = importance_to_heatmap(segments, importance)
    lr_target_box = hr_box_to_lr_box(target_box, scale)
    target_region = reconstruction.crop(target_box)

    figure, axes = plt.subplots(1, 4, figsize=(14, 4))
    axes[0].imshow(draw_target_box(lr_image, lr_target_box))
    axes[0].set_title('LR input')
    axes[0].axis('off')
    axes[1].imshow(draw_target_box(reconstruction, target_box))
    axes[1].set_title('IMDN output')
    axes[1].axis('off')
    axes[2].imshow(target_region)
    axes[2].set_title('Target region')
    axes[2].axis('off')
    axes[3].imshow(lr_image)
    axes[3].imshow(heatmap_lr, cmap='coolwarm', alpha=0.58, vmin=-1, vmax=1)
    axes[3].set_title(f'{method_name} importance')
    axes[3].axis('off')
    figure.tight_layout()
    figure.savefig(output_path, dpi=220, bbox_inches='tight')
    plt.close(figure)
    return output_path


## Step 6: LIME-style explanation

LIME creates many changed versions of the LR input, then fits a simple linear model to explain the score changes. The linear model is not replacing IMDN; it is only a local explanation tool.


In [ ]:
def explain_with_lime(score_masks, feature_count, sample_count):
    masks = rng.integers(0, 2, size=(sample_count, feature_count))
    masks[0, :] = 1
    scores = score_masks(masks)
    full_mask = np.ones((1, feature_count), dtype=int)
    distances = pairwise_distances(masks, full_mask, metric='cosine').reshape(-1)
    kernel_width = 0.75 * np.sqrt(feature_count)
    sample_weights = np.sqrt(np.exp(-(distances ** 2) / (kernel_width ** 2)))
    explanation_model = Ridge(alpha=1.0)
    explanation_model.fit(masks, scores, sample_weight=sample_weights)
    return explanation_model.coef_, float(scores[0]), float(np.mean(scores)), float(np.min(scores))


## Step 7: SHAP explanation

SHAP estimates each segment's contribution to the selected region score. It uses the same score function as LIME, so both methods explain the same question.


In [ ]:
def explain_with_shap(score_masks, feature_count, sample_count):
    background = np.zeros((1, feature_count), dtype=int)
    full_mask = np.ones((1, feature_count), dtype=int)
    explainer = shap.KernelExplainer(score_masks, background)
    values = explainer.shap_values(full_mask, nsamples=sample_count, silent=True)
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    full_score = float(score_masks(full_mask)[0])
    return values, full_score


## Step 8: Run final explanations

This is the main run. It produces one LIME figure and one SHAP figure for each selected case, then saves a CSV summary.


In [ ]:
records = []
checkpoint_cache = {}
model_cache = {}

for dataset, image_name, scale, region_type in FINAL_CASES:
    if scale not in checkpoint_cache:
        checkpoint_cache[scale] = download_official_imdn_checkpoint(CHECKPOINT_ROOT, scale)
    if scale not in model_cache:
        model_cache[scale] = load_pretrained_imdn(checkpoint_cache[scale], scale, DEVICE)
    model = model_cache[scale]

    hr_path = dataset_hr_directory(dataset, DATA_ROOT) / image_name
    lr_path = dataset_lr_directory(dataset, scale, DATA_ROOT) / image_name
    reference_hr = load_rgb_image(hr_path)
    lr_image = load_rgb_image(lr_path)
    validate_hr_lr_dimensions(reference_hr, lr_image, scale)

    target_box = centre_target_box(reference_hr, TARGET_HR_SIZE)
    segments = make_grid_segments(lr_image.width, lr_image.height, GRID_ROWS, GRID_COLUMNS)
    feature_count = GRID_ROWS * GRID_COLUMNS
    score_masks = mask_to_score_function(model, reference_hr, lr_image, segments, target_box)

    baseline_start = time.perf_counter()
    original_reconstruction = run_imdn_reconstruction(model, lr_image, reference_hr)
    original_psnr_y = score_target_region(reference_hr, original_reconstruction, target_box)
    baseline_seconds = time.perf_counter() - baseline_start

    case_slug = f'{dataset}_{Path(image_name).stem}_x{scale}'
    case_figure_root = FIGURE_ROOT / case_slug
    case_figure_root.mkdir(parents=True, exist_ok=True)

    lime_start = time.perf_counter()
    lime_importance, lime_full_score, lime_mean_score, lime_min_score = explain_with_lime(
        score_masks, feature_count, LIME_SAMPLES
    )
    lime_seconds = time.perf_counter() - lime_start
    lime_path = save_explanation_figure(
        lr_image,
        original_reconstruction,
        reference_hr,
        segments,
        lime_importance,
        'LIME',
        target_box,
        scale,
        case_figure_root / 'lime_importance_final.png',
    )

    shap_start = time.perf_counter()
    shap_importance, shap_full_score = explain_with_shap(score_masks, feature_count, SHAP_SAMPLES)
    shap_seconds = time.perf_counter() - shap_start
    shap_path = save_explanation_figure(
        lr_image,
        original_reconstruction,
        reference_hr,
        segments,
        shap_importance,
        'SHAP',
        target_box,
        scale,
        case_figure_root / 'shap_importance_final.png',
    )

    records.append({
        'dataset': dataset,
        'image': image_name,
        'scale': f'x{scale}',
        'region_type': region_type,
        'target_box_left': target_box[0],
        'target_box_top': target_box[1],
        'target_box_right': target_box[2],
        'target_box_bottom': target_box[3],
        'grid_rows': GRID_ROWS,
        'grid_columns': GRID_COLUMNS,
        'segment_count': feature_count,
        'lime_samples': LIME_SAMPLES,
        'shap_samples': SHAP_SAMPLES,
        'original_target_psnr_y': original_psnr_y,
        'lime_full_mask_psnr_y': lime_full_score,
        'lime_mean_perturbed_psnr_y': lime_mean_score,
        'lime_min_perturbed_psnr_y': lime_min_score,
        'lime_mean_drop_psnr_y': original_psnr_y - lime_mean_score,
        'shap_full_mask_psnr_y': shap_full_score,
        'baseline_seconds': baseline_seconds,
        'lime_seconds': lime_seconds,
        'shap_seconds': shap_seconds,
        'lime_figure': str(lime_path.relative_to(OUTPUT_ROOT)),
        'shap_figure': str(shap_path.relative_to(OUTPUT_ROOT)),
    })
    print(
        f"PASS: {dataset}/{image_name} x{scale}; "
        f"target PSNR-Y {original_psnr_y:.4f} dB; "
        f"LIME {lime_seconds:.1f}s; SHAP {shap_seconds:.1f}s"
    )

for model in model_cache.values():
    del model
torch.cuda.empty_cache()

summary_csv = write_results_csv(records, METRICS_ROOT / 'phase6_xai_final_summary.csv', overwrite=True)
print('Saved:', summary_csv)


## Step 9: Save settings

The settings file records exactly what was run. This matters because XAI results depend on choices such as grid size, sample count, and target region.


In [ ]:
settings = {
    'phase': 6,
    'run_name': 'imdn_xai_final_v1',
    'model': 'IMDN',
    'purpose': 'Final LIME and SHAP explanation maps for selected IMDN super-resolution cases.',
    'target_metric': 'PSNR-Y on centred HR target region',
    'cases': [
        {'dataset': dataset, 'image': image, 'scale': scale, 'region_type': region_type}
        for dataset, image, scale, region_type in FINAL_CASES
    ],
    'grid_rows': GRID_ROWS,
    'grid_columns': GRID_COLUMNS,
    'lime_samples': LIME_SAMPLES,
    'shap_samples': SHAP_SAMPLES,
    'target_hr_size': TARGET_HR_SIZE,
    'random_seed': RANDOM_SEED,
    'perturbation': 'replace hidden LR segments with Gaussian-blurred LR pixels, radius 3',
    'generated_at_utc': datetime.now(UTC).isoformat(),
}
settings_path = METRICS_ROOT / 'phase6_xai_final_settings.json'
settings_path.write_text(json.dumps(settings, indent=2) + '\n', encoding='utf-8')
print('Saved:', settings_path)


## Return the results

After running the notebook, return these files from `MyDrive/FYP_SR_Data/results/phase6/imdn_xai_final_v1/`:

1. `metrics/phase6_xai_final_summary.csv`
2. `metrics/phase6_xai_final_settings.json`
3. the `figures/` folder, or a zip of it

These are the files needed to write the final Phase 6 findings.
